<a href="https://colab.research.google.com/github/riballes/mobile-rag-firewall-/blob/main/experiments/phi_ner/notebooks/01_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHI NER Fine-tuning on Google Colab

Fine-tune BERT-like models to detect patient **names** and **addresses** in LLM responses, for use in the FW-L2 response firewall.

**Why this exists**: FW-L2 uses regex to catch SSN, phone, email, DOB — but regex can't detect free-form names ("Adah626 Klein929") or addresses ("308 Deckow Union, Pasco"). We fine-tune a small BERT model to fill this gap.

**What this notebook does:**
1. Loads NER training data from Weave (generated from Synthea patient records)
2. Trains 3 model architectures: DistilBERT (66M), BERT (110M), RoBERTa (125M)
3. Evaluates each on a held-out test set via Weave
4. Publishes the best model as a W&B artifact for automatic deployment in FW-L2

**Labels:** `O` (not PHI), `B-NAME` / `I-NAME` (patient name), `B-ADDRESS` / `I-ADDRESS` (address)

**Runtime**: GPU (T4) — Go to Runtime > Change runtime type > GPU

**Estimated time**: ~30-40 minutes for all 3 models

## 1. Setup

Install dependencies and authenticate with W&B.

**W&B API key**: Add `WANDB_API_KEY` to Colab Secrets (key icon in left sidebar) for automatic login. Get your key from [wandb.ai/authorize](https://wandb.ai/authorize).

In [1]:
# Start Time
import time
start_time = time.time()

In [2]:
!pip install -q transformers torch wandb weave accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.5/983.5 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 157.9 MB/s eta 0:00:00


In [3]:
import wandb
import weave
from google.colab import userdata

WANDB_PROJECT = "mobile-rag-firewall"

# Option A: Use Colab secrets (recommended)
# Go to the key icon in the left sidebar > add WANDB_API_KEY
try:
    wandb_key = userdata.get("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("Logged in via Colab secrets")
except Exception:
    # Option B: Manual login — paste your key when prompted
    # Get it from https://wandb.ai/authorize
    wandb.login()

weave.init(WANDB_PROJECT)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ricardo-morales-b to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged in via Colab secrets


weave: Logged in as Weights & Biases user: ricardo-morales-b.
weave: View Weave data at https://wandb.ai/ricardo-morales-b/mobile-rag-firewall/weave


In [4]:
# Check GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    print(f"Memory: {mem / 1024**3:.1f} GB")


GPU available: True
GPU: Tesla T4
Memory: 14.6 GB


## 2. Load Training Data

Training data is pulled from Weave (`phi-ner-train:latest`, `phi-ner-val:latest`, `phi-ner-test:latest`). These were generated locally by `uv run ner-generate` and published to Weave automatically.

Each example contains:
- `tokens`: list of words (e.g., `["Patient", "Adah626", "Klein929", "lives", "at", "Pasco"]`)
- `ner_tags`: BIO labels per token (e.g., `["O", "B-NAME", "I-NAME", "O", "O", "B-ADDRESS"]`)
- `has_entities`: whether this example contains any NAME/ADDRESS entities

**Data split**: 70% train / 15% val / 15% test

If Weave is unavailable, the cell falls back to manual file upload.

In [5]:
import json
from pathlib import Path
from google.colab import files

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Pull latest version from Weave (recommended)
try:
    train_dataset = weave.ref("phi-ner-train:latest").get()
    val_dataset = weave.ref("phi-ner-val:latest").get()
    test_dataset = weave.ref("phi-ner-test:latest").get()

    train_data = train_dataset.rows
    val_data = val_dataset.rows
    test_data = test_dataset.rows

    print(f"Loaded latest from Weave:")
    print(f"  Train: {len(train_data)} examples")
    print(f"  Val:   {len(val_data)} examples")
    print(f"  Test:  {len(test_data)} examples")
except Exception as e:
    print(f"Could not load from Weave: {e}")
    print("Upload train.json, val.json, test.json manually")
    uploaded = files.upload()
    with open("train.json") as f: train_data = json.load(f)
    with open("val.json") as f: val_data = json.load(f)
    with open("test.json") as f: test_data = json.load(f)


Loaded latest from Weave:
  Train: 8682 examples
  Val:   1861 examples
  Test:  1861 examples


## 3. Dataset and Model Setup

**PHINERDataset**: Converts our token + BIO tag data into the format HuggingFace Trainer expects. The key challenge is **subword alignment** — BERT tokenizes "Klein929" into `["Klein", "##929"]`, so we need to map our word-level labels to subword tokens:
- First subword of a word gets the word's label
- Continuation subwords (`##929`) get `I-` labels if the word is an entity, or `-100` (ignored in loss) otherwise

**compute_metrics**: Calculates token-level precision, recall, and F1 per entity type (NAME, ADDRESS) during training. This is reported after each epoch in W&B.

**Model configs**: Three architectures compared, each with tuned learning rates and batch sizes optimized for T4 16GB:
| Model | Params | LR | Batch | Why |
|-------|--------|-----|-------|-----|
| DistilBERT | 66M | 5e-5 | 32 | Smallest, fastest — best if task is easy |
| BERT | 110M | 3e-5 | 24 | Standard baseline |
| RoBERTa | 125M | 2e-5 | 24 | Often best for NER — but needs lower LR |

In [6]:
import numpy as np
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)

# NER label scheme
LABEL_LIST = ["O", "B-NAME", "I-NAME", "B-ADDRESS", "I-ADDRESS"]
LABEL_TO_ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID_TO_LABEL = {i: l for l, i in LABEL_TO_ID.items()}

# Model configurations — optimized for T4 16GB with FP16
MODEL_CONFIGS = {
    "distilbert": {"name": "distilbert-base-uncased", "lr": 5e-5, "epochs": 5, "batch_size": 32},
    "bert": {"name": "bert-base-uncased", "lr": 3e-5, "epochs": 5, "batch_size": 24},
    "roberta": {"name": "roberta-base", "lr": 2e-5, "epochs": 5, "batch_size": 24},
}

print(f"Labels: {LABEL_LIST}")
print(f"Models: {list(MODEL_CONFIGS.keys())}")

Labels: ['O', 'B-NAME', 'I-NAME', 'B-ADDRESS', 'I-ADDRESS']
Models: ['distilbert', 'bert', 'roberta']


In [7]:
class PHINERDataset(TorchDataset):
    """Token classification dataset for PHI NER."""

    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        tokens = example["tokens"]
        ner_tags = example["ner_tags"]

        encoding = self.tokenizer(
            tokens, is_split_into_words=True,
            truncation=True, max_length=self.max_length, padding=False,
        )

        word_ids = encoding.word_ids()
        labels = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                labels.append(-100)
            elif word_id != previous_word_id:
                label_str = ner_tags[word_id] if word_id < len(ner_tags) else "O"
                labels.append(LABEL_TO_ID.get(label_str, 0))
            else:
                label_str = ner_tags[word_id] if word_id < len(ner_tags) else "O"
                if label_str.startswith("B-"):
                    labels.append(LABEL_TO_ID.get("I-" + label_str[2:], 0))
                elif label_str.startswith("I-"):
                    labels.append(LABEL_TO_ID.get(label_str, 0))
                else:
                    labels.append(-100)
            previous_word_id = word_id

        encoding["labels"] = labels
        return {k: torch.tensor(v) for k, v in encoding.items()}

print("Dataset class ready.")

Dataset class ready.


In [8]:
def compute_metrics(eval_pred):
    """Compute precision, recall, F1 per entity type."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)

    true_labels, pred_labels = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                true_labels.append(l)
                pred_labels.append(p)

    true_labels = np.array(true_labels)
    pred_labels = np.array(pred_labels)
    metrics = {}

    entity_mask = true_labels != LABEL_TO_ID["O"]
    if entity_mask.sum() > 0:
        entity_correct = (pred_labels[entity_mask] == true_labels[entity_mask]).sum()
        metrics["entity_accuracy"] = float(entity_correct / entity_mask.sum())

    for entity in ["NAME", "ADDRESS"]:
        b_id = LABEL_TO_ID.get(f"B-{entity}", -1)
        i_id = LABEL_TO_ID.get(f"I-{entity}", -1)
        true_entity = np.isin(true_labels, [b_id, i_id])
        pred_entity = np.isin(pred_labels, [b_id, i_id])

        tp = (true_entity & pred_entity).sum()
        fp = (~true_entity & pred_entity).sum()
        fn = (true_entity & ~pred_entity).sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        metrics[f"precision_{entity}"] = precision
        metrics[f"recall_{entity}"] = recall
        metrics[f"f1_{entity}"] = f1

    entity_f1s = [metrics.get(f"f1_{e}", 0) for e in ["NAME", "ADDRESS"]]
    metrics["f1_macro"] = np.mean(entity_f1s)

    return metrics

print("Metrics function ready.")

Metrics function ready.


## 4. Training Function

`train_model()` handles the full cycle for one model:
1. Initialize W&B run with config + tags
2. Load pre-trained model + tokenizer from HuggingFace
3. Add a token classification head (5 output labels)
4. Train with HuggingFace `Trainer` — evaluates on val set after each epoch
5. Save the best checkpoint (based on `f1_macro`)
6. Log final metrics to W&B

**Key training arguments:**
- `eval_strategy="epoch"` — evaluate after every full pass through the data
- `load_best_model_at_end=True` — keeps the checkpoint with highest F1, not the last one
- `fp16=True` — half-precision training on GPU (2x faster, same accuracy)
- `EarlyStoppingCallback(patience=2)` — stops training if F1 doesn't improve for 2 consecutive epochs
- `dataloader_num_workers=2` — parallel data loading for better GPU utilization
- `report_to="wandb"` — training curves appear in W&B dashboard automatically

In [9]:
def train_model(model_key, train_data, val_data):
    """Train a single model configuration."""
    config = MODEL_CONFIGS[model_key]
    model_name = config["name"]
    output_dir = f"models/{model_key}"

    print(f"\n{'=' * 60}")
    print(f"  Training: {model_key} ({model_name})")
    print(f"  Train: {len(train_data)}, Val: {len(val_data)}")
    print(f"{'=' * 60}")

    wandb.init(
        project=WANDB_PROJECT,
        name=f"phi-ner-{model_key}",
        config={"model": model_name, "model_key": model_key,
                "learning_rate": config["lr"], "epochs": config["epochs"],
                "batch_size": config["batch_size"],
                "train_size": len(train_data), "val_size": len(val_data),
                "labels": LABEL_LIST},
        tags=["phi-ner", model_key, "colab"],
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForTokenClassification.from_pretrained(
        model_name, num_labels=len(LABEL_LIST),
        id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
    )

    train_dataset = PHINERDataset(train_data, tokenizer)
    val_dataset = PHINERDataset(val_data, tokenizer)
    data_collator = DataCollatorForTokenClassification(tokenizer)

    training_args = TrainingArguments(
        output_dir=output_dir,
        run_name=f"phi-ner-{model_key}",
        report_to="wandb",
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],
        learning_rate=config["lr"],
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=val_dataset,
        data_collator=data_collator, compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # Save best model
    best_dir = f"{output_dir}/best"
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)

    # Final evaluation
    eval_results = trainer.evaluate()
    print(f"\n  Final results for {model_key}:")
    for k, v in eval_results.items():
        if isinstance(v, float):
            print(f"    {k}: {v:.4f}")

    wandb.log({f"final/{k}": v for k, v in eval_results.items()})
    wandb.finish()

    return eval_results

print("Training function ready.")

Training function ready.


## 5. Train All 3 Models

Trains DistilBERT, BERT, and RoBERTa sequentially. Early stopping (patience=2) will halt training if F1 plateaus before the max 5 epochs.

After all 3 complete, a comparison table shows F1 macro, F1 per entity type. Look for:
- **F1 macro > 0.99**: The task is well-learned
- **F1 NAME vs F1 ADDRESS**: Names are typically easier (consistent patterns) than addresses (more variation)
- **DistilBERT >= others**: If the smallest model wins, the task doesn't need more capacity

In [10]:
all_results = {}

for model_key in ["distilbert", "bert", "roberta"]:
    results = train_model(model_key, train_data, val_data)
    all_results[model_key] = results

# Print comparison
print(f"\n{'=' * 60}")
print(f"  MODEL COMPARISON (validation set)")
print(f"{'=' * 60}")
print(f"  {'Model':<15} {'F1 Macro':>10} {'F1 NAME':>10} {'F1 ADDR':>10}")
print(f"  {'-'*15} {'-'*10} {'-'*10} {'-'*10}")
for model_key, results in all_results.items():
    f1_macro = results.get("eval_f1_macro", 0)
    f1_name = results.get("eval_f1_NAME", 0)
    f1_addr = results.get("eval_f1_ADDRESS", 0)
    print(f"  {model_key:<15} {f1_macro:>10.4f} {f1_name:>10.4f} {f1_addr:>10.4f}")
print(f"{'=' * 60}")


  Training: distilbert (distilbert-base-uncased)
  Train: 8682, Val: 1861


wandb: Initializing weave.


Output()

weave: Logged in as Weights & Biases user: ricardo-morales-b.
weave: View Weave data at https://wandb.ai/ricardo-morales-b/mobile-rag-firewall/weave
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Final results for roberta:
    eval_loss: 0.0008
    eval_entity_accuracy: 0.9969
    eval_precision_NAME: 1.0000
    eval_recall_NAME: 1.0000
    eval_f1_NAME: 1.0000
    eval_precision_ADDRESS: 0.9946
    eval_recall_ADDRESS: 0.9943
    eval_f1_ADDRESS: 0.9945
    eval_f1_macro: 0.9972
    eval_runtime: 7.0908
    eval_samples_per_second: 262.4520
    eval_steps_per_second: 11.0000
    epoch: 5.0000


eval/entity_accuracy,▁▅▆███
eval/f1_ADDRESS,▁▇▆███
eval/f1_NAME,▁█████
eval/f1_macro,▁▇▆███
eval/loss,█▂▂▁▁▁
eval/precision_ADDRESS,▁█▄███
eval/precision_NAME,▁█████
eval/recall_ADDRESS,▁▇████
eval/recall_NAME,▁▁▁▁▁▁
eval/runtime,█▆▁▄▅▅
+20,...



  MODEL COMPARISON (validation set)
  Model             F1 Macro    F1 NAME    F1 ADDR
  --------------- ---------- ---------- ----------
  distilbert          0.9972     1.0000     0.9944
  bert                0.9975     1.0000     0.9950
  roberta             0.9972     1.0000     0.9945


## 6. Weave Evaluation on Test Set

Runs each trained model through a formal **Weave Evaluation** on the held-out test set. This is different from the training eval:
- Training eval: token-level metrics on the validation set
- Weave eval: entity-level detection rates on the **test set** (never seen during training)

The `NERModel` wraps each trained model as a Weave Model. The `ner_scorer` measures:
- `name_detected`: % of true NAME entities found
- `address_detected`: % of true ADDRESS entities found
- `name_false_positive`: % of non-NAME tokens incorrectly labeled as NAME
- `address_false_positive`: % of non-ADDRESS tokens incorrectly labeled as ADDRESS

Results appear on the W&B Weave dashboard for leaderboard comparison.

In [11]:
from transformers import pipeline as hf_pipeline


class NERModel(weave.Model):
    model_key: str = ""
    model_path: str = ""
    _pipeline: object = None

    def _ensure_loaded(self):
        if self._pipeline is None:
            self._pipeline = hf_pipeline(
                "ner", model=self.model_path, tokenizer=self.model_path,
                aggregation_strategy="simple",
            )

    @weave.op
    def predict(self, tokens: list, ner_tags: list) -> dict:
        self._ensure_loaded()
        text = " ".join(tokens)
        results = self._pipeline(text)

        pred_entities = [
            {"entity": r["entity_group"], "word": r["word"], "score": float(r["score"])}
            for r in results if r["entity_group"] != "O"
        ]

        return {
            "predicted_entities": pred_entities,
            "pred_name_count": sum(1 for e in pred_entities if e["entity"] == "NAME"),
            "pred_address_count": sum(1 for e in pred_entities if e["entity"] == "ADDRESS"),
            "true_name_count": sum(1 for t in ner_tags if t in ("B-NAME", "I-NAME")),
            "true_address_count": sum(1 for t in ner_tags if t in ("B-ADDRESS", "I-ADDRESS")),
            "has_entities": any(t != "O" for t in ner_tags),
        }


@weave.op
def ner_scorer(output: dict) -> dict:
    pred_names = output.get("pred_name_count", 0)
    pred_addrs = output.get("pred_address_count", 0)
    true_names = output.get("true_name_count", 0)
    true_addrs = output.get("true_address_count", 0)

    name_detected = 1.0 if (true_names > 0 and pred_names > 0) else (1.0 if true_names == 0 else 0.0)
    addr_detected = 1.0 if (true_addrs > 0 and pred_addrs > 0) else (1.0 if true_addrs == 0 else 0.0)
    name_fp = 1.0 if (true_names == 0 and pred_names > 0) else 0.0
    addr_fp = 1.0 if (true_addrs == 0 and pred_addrs > 0) else 0.0

    return {
        "name_detected": name_detected,
        "address_detected": addr_detected,
        "name_false_positive": name_fp,
        "address_false_positive": addr_fp,
    }

print("Weave evaluation ready.")

Weave evaluation ready.


In [12]:
# Run Weave evaluation for each model on the test set
test_dataset = weave.Dataset(name="phi-ner-test", rows=test_data)
weave.publish(test_dataset)

for model_key in ["distilbert", "bert", "roberta"]:
    model_path = f"models/{model_key}/best"
    if not Path(model_path).exists():
        print(f"Skipping {model_key} — not trained")
        continue

    print(f"\nEvaluating {model_key} on test set...")

    model = NERModel(
        name=f"phi-ner-{model_key}",
        model_key=model_key,
        model_path=model_path,
    )

    evaluation = weave.Evaluation(
        name=f"ner-eval-{model_key}",
        dataset=test_dataset,
        scorers=[ner_scorer],
        metadata={"model_key": model_key, "source": "colab"},
    )

    results = await evaluation.evaluate(model)
    print(f"  {model_key}: {results}")

weave: 📦 Published to https://wandb.ai/ricardo-morales-b/mobile-rag-firewall/weave/objects/phi-ner-test/versions/BFy1hXzorr0LfqqNE1JI8nEROhzMxLAwHZxHvFD2b88



Evaluating distilbert on test set...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

weave: 🍩 https://wandb.ai/ricardo-morales-b/mobile-rag-firewall/r/call/019d7feb-b4b2-7eef-8174-176875183ff9


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

weave: Evaluated 1 of 1861 examples
weave: Evaluated 2 of 1861 examples
weave: Evaluated 3 of 1861 examples
weave: Evaluated 4 of 1861 examples
weave: Evaluated 5 of 1861 examples
weave: Evaluated 6 of 1861 examples
weave: Evaluated 7 of 1861 examples
weave: Evaluated 8 of 1861 examples
weave: Evaluated 9 of 1861 examples
weave: Evaluated 10 of 1861 examples
weave: Evaluated 11 of 1861 examples
weave: Evaluated 12 of 1861 examples
weave: Evaluated 13 of 1861 examples
weave: Evaluated 14 of 1861 examples
weave: Evaluated 15 of 1861 examples
weave: Evaluated 16 of 1861 examples
weave: Evaluated 17 of 1861 examples
weave: Evaluated 18 of 1861 examples
weave: Evaluated 19 of 1861 examples
weave: Evaluated 20 of 1861 examples
weave: Evaluated 21 of 1861 examples
weave: Evaluated 22 of 1861 examples
weave: Evaluated 23 of 1861 examples
weave: Evaluated 24 of 1861 examples
weave: Evaluated 25 of 1861 examples
weave: Evaluated 26 of 1861 examples
weave: Evaluated 27 of 1861 examples
weave: Eva

  roberta: {'output': {'pred_name_count': {'mean': 0.29607737775389575}, 'pred_address_count': {'mean': 1.3036002149382053}, 'true_name_count': {'mean': 0.6055883933369156}, 'true_address_count': {'mean': 1.4653412144008597}, 'has_entities': {'true_count': 738, 'true_fraction': 0.39656098871574424}}, 'ner_scorer': {'name_detected': {'mean': 1.0}, 'address_detected': {'mean': 0.9978506179473401}, 'name_false_positive': {'mean': 0.0010746910263299302}, 'address_false_positive': {'mean': 0.06663084363245567}}, 'model_latency': {'mean': 1.7263512634449527}}


## 7. Publish Best Model

Publish the best model to W&B as a model artifact (`phi-ner-model`). FW-L2 in the backend will pull it automatically via `run.use_artifact("phi-ner-model:latest")` — no manual download or file transfer needed.

**Choose the best model** by changing `BEST_MODEL` based on the comparison table from Step 5. Consider:
- **Highest F1 macro**: Best overall accuracy
- **Lowest false positive rate**: Least over-redaction of clean text
- **Smallest size** (DistilBERT): Fastest inference, smallest footprint

If you want a local copy too, uncomment the zip download lines at the bottom.

In [13]:
import shutil
import os

# Choose the best model based on results above
BEST_MODEL = "distilbert"  # Change this based on results
best_model_path = f"models/{BEST_MODEL}/best"

# ── Publish as W&B Model Artifact ──
# This makes the model available as a proper artifact that can be downloaded
# and loaded with AutoModelForTokenClassification.from_pretrained()

run = wandb.init(
    project=WANDB_PROJECT,
    name=f"publish-phi-ner-{BEST_MODEL}",
    job_type="publish-model",
    tags=["phi-ner", BEST_MODEL, "publish"],
)

artifact = wandb.Artifact(
    name="phi-ner-model",
    type="model",
    description=f"Best PHI NER model ({BEST_MODEL}) for detecting NAME and ADDRESS entities",
    metadata={"model_key": BEST_MODEL, "base_model": MODEL_CONFIGS[BEST_MODEL]["name"]},
)
artifact.add_dir(best_model_path)
run.log_artifact(artifact)
run.finish()

print(f"Published model artifact: phi-ner-model:latest")
print(f"Download with:")
print(f"  run = wandb.init(project='{WANDB_PROJECT}')")
print(f"  artifact = run.use_artifact('phi-ner-model:latest')")
print(f"  model_dir = artifact.download()")
print(f"  model = AutoModelForTokenClassification.from_pretrained(model_dir)")

# ── Optional: Download zip for local use ──
# Uncomment the lines below if you also want a local copy
# shutil.make_archive(f"phi-ner-{BEST_MODEL}", "zip", best_model_path)
# files.download(f"phi-ner-{BEST_MODEL}.zip")

wandb: Initializing weave.


Output()

weave: Flushing 3 pending tasks...


weave: Logged in as Weights & Biases user: ricardo-morales-b.
weave: View Weave data at https://wandb.ai/ricardo-morales-b/mobile-rag-firewall/weave
wandb: Adding directory to artifact (models/distilbert/best)... Done. 0.7s


Published model artifact: phi-ner-model:latest
Download with:
  run = wandb.init(project='mobile-rag-firewall')
  artifact = run.use_artifact('phi-ner-model:latest')
  model_dir = artifact.download()
  model = AutoModelForTokenClassification.from_pretrained(model_dir)


In [14]:
# End Time
end_time = time.time()
total_time = end_time - start_time

# Convert to minutes and seconds
mins, secs = divmod(total_time, 60)
print(f"Total Notebook Execution Time: {int(mins)}m {int(secs)}s")

Total Notebook Execution Time: 35m 20s
